## PyTorch를 활용한 이진 분류 모델 구현과 최적화

이번 실습에서는 PyTorch를 활용하여 간단한 2차원 이진 분류 모델을 구현하고 학습시켜 볼 것입니다.

아래 지시사항의 안내를 따라 코드를 완성하세요.

### 0. 라이브러리 불러오기

필요한 PyTorch 라이브러리를 불러옵니다.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

answer = {}

### 1. 데이터 탐색 및 전처리 (0점)

실습에 사용할 `data.csv` 파일을 불러와 데이터 구조를 확인합니다.

이 데이터는 두 개의 입력값 `x1`, `x2`와 정답 라벨 `label`로 구성되어 있습니다.  
클래스 0은 `(0, 0)` 근처, 클래스 1은 `(3, 3)` 근처에 모여 있으며, 테스트 구간 일부에는 라벨 노이즈가 포함되어 있습니다. 모델은 두 좌표값을 보고 해당 데이터가 어느 클래스에 속하는지 예측합니다.

In [2]:
# 준비된 데이터 파일(data.csv)을 불러옵니다.
df = pd.read_csv("data.csv")
print("데이터 shape:", df.shape)
df.head()

데이터 shape: (1000, 3)


,x1,x2,label
0,-0.141811,-0.276341,0
1,2.912322,2.844913,1
2,2.414766,1.790237,1
3,3.313839,2.735147,1
4,0.957831,-0.822531,0


In [3]:
# 불러온 데이터를 텐서로 변환합니다.
X = torch.tensor(df[["x1", "x2"]].values, dtype=torch.float32)
y = torch.tensor(df["label"].values, dtype=torch.long)
print("X:", X.shape, "y:", y.shape)

X: torch.Size([1000, 2]) y: torch.Size([1000])


### 2. DataLoader 구성 (20점)

아래 코드의 `None`을 채워 학습, 검증, 테스트용 DataLoader를 구성하세요.

이미 위에서 데이터를 다음과 같이 나누어 두었습니다.

- 학습 데이터: `X_train`, `y_train`
- 검증 데이터: `X_val`, `y_val`
- 테스트 데이터: `X_test`, `y_test`

각 DataLoader는 다음 조건을 만족해야 합니다.

- 입력 데이터와 정답 라벨을 `TensorDataset`으로 묶습니다.
- 모든 DataLoader의 `batch_size`는 `32`로 설정합니다.
- 학습용 DataLoader는 매 epoch마다 데이터를 섞을 수 있도록 `shuffle=True`로 설정합니다.
- 검증용, 테스트용 DataLoader는 평가 결과가 일정하게 유지되도록 `shuffle=False`로 설정합니다.

In [4]:
# 1-1. 데이터 분할 — 앞에서부터 800 / 100 / 100 으로 나눕니다.
X_train, y_train = X[:800], y[:800]         # 앞에서부터 800개
X_val,   y_val   = X[800:900], y[800:900]   # 그다음 100개
X_test,  y_test  = X[900:], y[900:]         # 마지막 100개

# 1-2. DataLoader 구성
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),   batch_size=32, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test),  batch_size=32, shuffle=False)

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
answer["Q2-1"] = len(train_loader.dataset)
answer["Q2-2"] = len(val_loader.dataset)
answer["Q2-3"] = len(test_loader.dataset)
answer["Q2-4"] = train_loader.batch_size
answer["Q2-5"] = val_loader.batch_size
answer["Q2-6"] = test_loader.batch_size
answer["Q2-7"] = str(type(train_loader.sampler))
answer["Q2-8"] = str(type(val_loader.sampler))
answer["Q2-9"] = str(type(test_loader.sampler))

### 3-1. 모델 정의 (10점)

아래 코드의 `None`을 채워 3개의 Linear 레이어를 가진 MLP 모델을 완성하세요.

모델 구조는 다음 조건을 만족해야 합니다.

- 입력 차원은 `2`입니다. (`x1`, `x2`)
- 첫 번째 은닉층의 출력 차원은 `16`입니다.
- 두 번째 은닉층의 출력 차원은 `8`입니다.
- 출력 차원은 `2`입니다. (`label` 0 또는 1)
- 은닉층 뒤에는 `ReLU` 활성화 함수를 사용합니다.
- 출력층 뒤에는 `Softmax`를 추가하지 않습니다.  
  `CrossEntropyLoss`가 내부적으로 softmax 처리를 포함하기 때문입니다.

`forward()`에서는 입력 `x`가 다음 순서로 지나가도록 코드를 완성하세요.

`fc1 → relu1 → fc2 → relu2 → fc3`

In [5]:
# 2. 모델 정의
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)   
        self.relu1 = nn.ReLU()                  
        self.fc2 = nn.Linear(16, 8)   
        self.relu2 = nn.ReLU()                  
        self.fc3 = nn.Linear(8, 2)   
    def forward(self, x):
        x = self.relu1(self.fc1(x))  
        x = self.relu2(self.fc2(x))      
        x = self.fc3(x)               
        return x
model = MLP()

model = MLP()

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
torch.save(model.state_dict(), "model.pth")
answer["Q3-1-A"] = [model.fc1.in_features, model.fc1.out_features]
answer["Q3-1-B"] = [model.fc2.in_features, model.fc2.out_features]
answer["Q3-1-C"] = [model.fc3.in_features, model.fc3.out_features]
answer["Q3-1-D"] = str(type(model.relu1))
answer["Q3-1-E"] = str(type(model.relu2))

### 3-2. 손실 함수와 최적화 함수 구성 (10점)

학습에 필요한 **손실 함수**와 **최적화 함수**를 만듭니다. 아래 조건을 만족하도록 각 줄의 `None`을 채우세요.

- loss는 `CrossEntropyLoss`를 사용합니다.
- optimizer는 `Adam`를 사용합니다.
- 학습률은 0.001로 설정합니다.

In [6]:
# 3. 손실 함수 / 최적화 함수
criterion = nn.CrossEntropyLoss()                                
optimizer = optim.Adam(model.parameters(), lr=0.001)      

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
answer["Q3-2-A"] = str(type(criterion))
answer["Q3-2-B"] = optimizer.state_dict()

### 4. 모델 학습 (30점)

아래 코드의 `None`을 채워 모델 학습 코드를 완성하세요.

학습은 다음 조건을 만족해야 합니다.

- 전체 학습 횟수 `epochs`는 `100`으로 설정합니다.
- `train_loader`에서 배치 단위로 데이터를 꺼내 학습합니다.
- 각 배치마다 다음 순서로 학습을 진행합니다.

    1. 이전 배치에서 계산된 gradient를 초기화합니다.
    2. 모델에 입력 데이터를 넣어 예측값을 계산합니다.
    3. 예측값과 정답 라벨을 이용해 loss를 계산합니다.
    4. `backward()`로 gradient를 계산합니다.
    5. optimizer를 이용해 모델 파라미터를 업데이트합니다.

배치에서 꺼낸 `batch_x`, `batch_y`는 각각 입력 데이터와 정답 라벨을 의미합니다.

In [7]:
# 4. 모델 학습
epochs = 100
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()   
        outputs = model(batch_x)          
        loss = criterion(outputs, batch_y) 
        loss.backward()        
        optimizer.step()

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
torch.save(model.state_dict(), "model.pth")
answer["Q4"] = {"epochs": epochs}

### 5. 모델 평가 (30점)

아래 코드의 `None`을 채워 테스트 데이터에 대한 평가 코드를 완성하세요.

평가는 다음 조건을 만족해야 합니다.

- 평가 시에는 `torch.no_grad()`를 사용해 gradient 계산을 끕니다.
- `test_loader`에서 배치 단위로 데이터를 꺼내 모델의 예측값을 계산합니다.
- 각 배치의 loss를 누적한 뒤, 전체 배치 수로 나누어 평균 loss를 구합니다.
- 예측 클래스는 `outputs.argmax(1)`을 사용해 구합니다.
- 맞힌 개수를 전체 테스트 데이터 개수로 나누어 정확도 `test_acc`를 계산합니다.

최종적으로 다음 두 변수에 값을 저장해야 합니다.

- `test_loss`: 테스트 데이터의 평균 손실값
- `test_acc`: 테스트 데이터의 정확도

In [14]:
# 5. 모델 평가
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        test_loss += loss.item() * batch_x.size(0)   # 배치 크기만큼 가중치
        preds = outputs.argmax(1)
        correct += (preds == batch_y).sum().item()
test_loss = test_loss / len(test_loader.dataset)      # 전체 샘플 수로 나눔
test_acc  = correct / len(test_loader.dataset)

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
print(f"Test accuracy: {test_acc:.3f}")
answer["Q5-1"] = test_loss
answer["Q5-2"] = test_acc

Test accuracy: 0.960


### 제출

모든 문제를 해결하셨으면 아래 코드를 실행해서 결과를 저장한 후, 우측 상단의 '제출' 버튼을 눌러서 코드를 제출해주세요.

In [15]:
# Export data
import json
with open("submission.json", "w") as f:
    json.dump(answer, f)